# Software Requirement Specification (SRS) & Project Documentation

**Project Title:** SyncTask – Cross-Platform Visual Urgency To-Do Ecosystem

**Target Architecture:** MERN Stack + React Native + Chrome Extension (Manifest V3)

**Author:** Akshat Raj

**Date:** May 2026

---

## 1. Executive Summary & Project Objectives

SyncTask is a multi-client, unified productivity ecosystem engineered to solve the problem of "deadline blindness." Traditional to-do apps rely on passive notification lists. SyncTask introduces **Visual Urgency UI**, where task states dynamically shift color along a gradient from **Green to Red** as real-time deadlines approach, factoring in a user-selected severity matrix.

The project establishes a centralized Node.js/Express backend coupled with a flexible MongoDB database, servicing two synchronized clients simultaneously:

1. An **Android Application** optimized for mobile access.
2. A **Chrome Extension** for desktop workflow integration.

---

## 2. System Architecture

### 2.1 Technical Stack Matrix

* **Database Tier:** MongoDB (NoSQL document store for high-velocity JSON mutations)
* **Backend Application Tier:** Node.js + Express.js REST API + Mongoose ORM
* **Mobile Client Tier:** React Native + NativeWind (Tailwind CSS engine)
* **Browser Client Tier:** React.js + Vite + Web Extensions Core API (Manifest V3)
* **Authentication Mechanism:** Stateless JWT (JSON Web Tokens) cross-shared via storage engines.

### 2.2 System Component Topology

```
                  ┌──────────────────────────────┐
                  │   Chrome Extension (React)   │
                  │   - Cache: chrome.storage    │
                  └──────────────┬───────────────┘
                                 │
                                 │ HTTPS / REST API (JWT Auth)
                                 ▼
┌──────────────────┐      ┌──────────────┐      ┌─────────────────┐
│ Android Device   ├─────►│  Node.js     ├─────►│   MongoDB       │
│ (React Native)   │      │  Express API │      │   Database      │
└──────────────────┘      └──────────────┘      └─────────────────┘

```

---

## 3. Functional Requirements

### 3.1 Core Data Schema Requirements

Every task document captured by the database system must strictly map to the following functional properties:

* **Task Title:** String (Max 100 characters, required).
* **Description:** String (Max 500 characters, optional).
* **Deadline:** ISO 8601 UTC Timestamp (Linked to a native Calendar Date/Time Picker).
* **Severity Matrix:** Enum value matching `['low', 'mid', 'high']`.
* **Completion State:** Boolean flag (`isCompleted`).

### 3.2 The Visual Urgency & Color-Shift Engine

The UI components must continuously compute the delta between the current client machine time ($T_{\text{now}}$) and the task deadline timestamp ($T_{\text{due}}$).

$$\Delta T = T_{\text{due}} - T_{\text{now}}$$

The background color classification shifts based on the remaining hours:

| Remaining Time ($\Delta T$) | Base Color State | UI Class / Behavior |
| --- | --- | --- |
| $\Delta T > 48 \text{ hours}$ | **Green** | Safe state |
| $24 \text{ hours} < \Delta T \le 48 \text{ hours}$ | **Yellow** | Warning state |
| $6 \text{ hours} < \Delta T \le 24 \text{ hours}$ | **Orange** | Escalated urgency state |
| $\Delta T \le 6 \text{ hours}$ | **Red** | Critical state + Pulse CSS Animation |
| $\Delta T \le 0 \text{ hours}$ | **Black** | Overdue / Expired status |

*Exception Override Logic:* If a task's `severity` parameter is flagged as **High**, it automatically skips the **Green** phase and manifests immediately as **Orange**, cascading to **Red** at the 12-hour mark.

---

## 4. Backend Implementation Plan (Node.js & MongoDB)

### 4.1 Database Design (Models)

The system requires a strict schema definition for Users and Tasks via Mongoose.

```javascript
// models/Task.js
const mongoose = require('mongoose');

const TaskSchema = new mongoose.Schema({
  userId: { type: mongoose.Schema.Types.ObjectId, ref: 'User', required: true },
  title: { type: String, required: true, trim: true },
  description: { type: String, trim: true },
  deadline: { type: Date, required: true },
  severity: { type: String, enum: ['low', 'mid', 'high'], default: 'low' },
  isCompleted: { type: Boolean, default: false }
}, { timestamps: true });

module.exports = mongoose.model('Task', TaskSchema);

```

### 4.2 REST API Endpoint Specifications

| HTTP Method | Resource Route | Authentication | Payload Requirements | Description |
| --- | --- | --- | --- | --- |
| `POST` | `/api/auth/register` | Public | `{ email, password }` | Registers a new account profile. |
| `POST` | `/api/auth/login` | Public | `{ email, password }` | Authenticates user; returns signed JWT. |
| `GET` | `/api/tasks` | Private (JWT) | None | Fetches all tasks assigned to authenticated `userId`. |
| `POST` | `/api/tasks` | Private (JWT) | `{ title, description, deadline, severity }` | Creates a new task instance. |
| `PUT` | `/api/tasks/:id` | Private (JWT) | `{ isCompleted: true }` | Updates targeted task status attributes. |
| `DELETE` | `/api/tasks/:id` | Private (JWT) | None | Hard deletes task from database collection. |

---

## 5. Client Applications Architectural Blueprint

### 5.1 Chrome Extension Configuration (Manifest V3)

To ensure the Chrome Extension can make API requests and spawn notifications, the file system configuration must be mapped as follows:

```json
{
  "manifest_version": 3,
  "name": "SyncTask Extension",
  "version": "1.0.0",
  "description": "Visual Urgency To-Do Engine synced with mobile.",
  "permissions": [
    "storage",
    "notifications",
    "alarms"
  ],
  "host_permissions": [
    "http://localhost:5000/*"
  ],
  "action": {
    "default_popup": "index.html"
  },
  "background": {
    "service_worker": "background.js",
    "type": "module"
  }
}

```

### 5.2 React Shared Color Utility (Clean Code Principle)

Using a unified functional asset across the Extension popup UI and the React Native view keeps code clean and testable:

```javascript
export const computeUrgencyTailwind = (deadlineStr, severity) => {
  const timeLeftMs = new Date(deadlineStr) - new Date();
  const hoursLeft = timeLeftMs / (1000 * 60 * 60);

  if (hoursLeft <= 0) return 'bg-neutral-900 text-white'; // Overdue
  if (hoursLeft <= 6) return 'bg-red-500 animate-pulse text-white'; // Critical
  
  // High severity automatically upgrades warning window
  if (hoursLeft <= 12 && severity === 'high') return 'bg-red-500 animate-pulse text-white';
  if (hoursLeft <= 24 || severity === 'high') return 'bg-orange-500 text-white';
  if (hoursLeft <= 48) return 'bg-yellow-400 text-slate-900';
  
  return 'bg-green-500 text-white'; // Safe
};

```

---

## 6. Critical Technical Implementation Blockers & Strategies

### 6.1 Cross-Origin Resource Sharing (CORS) Bottlenecks

* **The Issue:** Chrome Extensions render from an internal isolated origin address (`chrome-extension://<EXTENSION_ID>`). Standard API security profiles automatically block this incoming traffic string.
* **Mitigation Strategy:** Configure the Express backend to dynamically parse origins.
```javascript
app.use(cors({
  origin: true, // Dynamically allows the incoming extension identifier string
  credentials: true
}));

```



### 6.2 Mobile Operating System Aggressive Power Management

* **The Issue:** Android devices invoke deep sleep states which freeze background execution loops within React Native. This prevents the deadline verification algorithm from triggering notifications.
* **Mitigation Strategy:** Do not compute time changes inside background loops on the phone. Instead, register real-time system alerts directly into the native system layer using **Firebase Cloud Messaging (FCM)** down-streamed from the backend, or leverage local background workers via Android `AlarmManager`.

### 6.3 Stateless Session Synchronization

* **The Issue:** Maintaining consistent authentication states simultaneously across independent browser storage and mobile keychains.
* **Mitigation Strategy:** Implement stateless token authorization. Upon user authentication, issue a long-lasting JWT token. The Chrome Extension stores this token in `chrome.storage.local`, while the React Native app maps it to `EncryptedStorage`. Both pass the authorization token inside the standard header wrapper:
```
Authorization: Bearer <TOKEN>

```



---

## 7. Timeline & Phased Project Milestones

* **Phase 1 (Sprint 1 - 4 Days):** Establish Express core engine routing paths. Implement Mongoose models and test JWT session responses using API clients.
* **Phase 2 (Sprint 2 - 5 Days):** Construct React Native view panels, layout calendar elements, and connect token-based storage pipelines to the API.
* **Phase 3 (Sprint 3 - 4 Days):** Scaffold the Manifest V3 Extension container file structure. Configure build compilation tooling for public folder directories and script file handling.
* **Phase 4 (Sprint 4 - 3 Days):** Implement cross-client sync testing, simulate visual color changes by modifying time windows, and compile placement review materials.